# Weekly player descriptions

Run the standard results pipeline, generate the prior-season ESPN scorer data, then run this notebook from the repository root:

```powershell
python 1_prepare_forms.py
python 2_output_results.py
python 3_generate_espn_data.py
```

This notebook performs the player-description data preparation and generation in one place. It reads the player reference, weekly goal files, ownership report, and `data/intermediate/<data_source>/espn_goals.csv`; it validates and prepares facts, then asks local Ollama to write the descriptions. It only uses facts verified for that player, rejects unsupported goal totals and scoring ranks, and overwrites `data/output/<data_source>/player_descriptions.csv` only after a successful complete run. `MAX_PLAYERS_TO_GENERATE` can limit a run; a limited run overwrites the output with that subset.

Install and start Ollama locally, then run `ollama pull llama3.2:3b` once before generating descriptions. The notebook makes requests only to `localhost`; no hosted model API is used.

In [2]:
from datetime import datetime
from pathlib import Path

import pandas as pd

from config import data_source
from utils.espn_utils import LAST_SEASON

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config.py").is_file():
    raise RuntimeError("Run this notebook with the repository root as the working directory.")

MODEL = "llama3.2:3b"
OLLAMA_URL = "http://localhost:11434/api/generate"
MAX_PLAYERS_TO_GENERATE = None  # Set to None to generate every eligible player.
DATA_DIR = PROJECT_ROOT / "data"
INPUT_DIR = DATA_DIR / "input" / data_source
INTERMEDIATE_DIR = DATA_DIR / "intermediate" / data_source
OUTPUT_DIR = DATA_DIR / "output" / data_source
OUTPUT_PATH = OUTPUT_DIR / "player_descriptions.csv"


In [3]:
import json
import math
import numbers
import re
import urllib.error
import urllib.request
from collections.abc import Mapping
from pathlib import Path
from typing import Any

import pandas as pd

MAX_DESCRIPTION_WORDS = 40
DESCRIPTION_VALIDATOR_VERSION = 10
MAX_VALIDATION_ATTEMPTS = 3
GOAL_FILE_PATTERN = re.compile(r"GW_(\d+)")
NUMBER_WORDS = {
    "zero": 0, "one": 1, "single": 1, "once": 1, "two": 2, "twice": 2,
    "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
    "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12,
    "thirteen": 13, "fourteen": 14, "fifteen": 15, "sixteen": 16,
    "seventeen": 17, "eighteen": 18, "nineteen": 19, "twenty": 20,
    "thirty": 30, "forty": 40, "first": 1, "second": 2, "third": 3,
    "fourth": 4, "fifth": 5, "sixth": 6, "seventh": 7, "eighth": 8,
    "ninth": 9, "tenth": 10,
}
NUMBER_TOKEN = r"(?:\d+(?:st|nd|rd|th)?|" + "|".join(
    sorted(NUMBER_WORDS, key=len, reverse=True)
) + r")(?:-(?:one|two|three|four|five|six|seven|eight|nine))?"
GOAL_COUNT_TOKEN = r"(?:no|a|an|" + NUMBER_TOKEN + r")"
GOAL_COUNT_PATTERN = re.compile(
    r"(?P<count>" + GOAL_COUNT_TOKEN + r")\s+(?:league\s+)?goals?\b",
    re.IGNORECASE,
)
RANK_CLAIM_PATTERN = re.compile(
    r"\brank(?:ed|ing|s)?\s+(?:at\s+)?(?P<rank>" + NUMBER_TOKEN + r")\b"
    r"|\b(?P<highest>first|second|third|fourth|fifth|\d+(?:st|nd|rd|th))"
    r"[- ]highest\b",
    re.IGNORECASE,
)
def _read_goal_history(goals_dir: Path, player_ids: set[str]) -> tuple[dict[str, list[int]], int]:
    files_by_gameweek = {}
    for path in goals_dir.glob("GW_*.csv"):
        match = GOAL_FILE_PATTERN.fullmatch(path.stem)
        if match is None:
            raise ValueError(f"Unexpected gameweek filename: {path.name}")
        gameweek = int(match.group(1))
        if gameweek in files_by_gameweek:
            raise ValueError(f"More than one file found for GW {gameweek}.")
        files_by_gameweek[gameweek] = path
    if not files_by_gameweek:
        raise FileNotFoundError(f"No GW_<number>.csv files found in {goals_dir}")
    latest_gameweek = max(files_by_gameweek)
    expected_gameweeks = set(range(1, latest_gameweek + 1))
    if set(files_by_gameweek) != expected_gameweeks:
        missing = sorted(expected_gameweeks - set(files_by_gameweek))
        raise ValueError(f"Gameweek files are not consecutive; missing: {missing}")

    goals_by_player = {player_id: [0] * latest_gameweek for player_id in player_ids}
    for gameweek, path in sorted(files_by_gameweek.items()):
        goals = pd.read_csv(path, dtype={"player_id": "string"})
        if "player_id" not in goals.columns:
            raise ValueError(f"{path.name} must contain a player_id column.")
        if "goals" not in goals.columns:
            goals["goals"] = 1
        goals["player_id"] = goals["player_id"].str.strip()
        if goals["player_id"].isna().any() or goals["player_id"].eq("").any():
            raise ValueError(f"{path.name} contains an empty player_id.")
        if goals["player_id"].duplicated().any():
            raise ValueError(f"{path.name} contains duplicate player_id values.")
        goals["goals"] = goals["goals"].replace(r"^\s*$", pd.NA, regex=True).fillna(1)
        numeric_goals = pd.to_numeric(goals["goals"], errors="coerce")
        if numeric_goals.isna().any() or (numeric_goals < 0).any() or (numeric_goals % 1 != 0).any():
            raise ValueError(f"{path.name} contains an invalid goals value.")
        unknown_ids = set(goals["player_id"]) - player_ids
        if unknown_ids:
            raise ValueError(f"{path.name} contains unknown player IDs: {sorted(unknown_ids)}")
        for player_id, count in zip(goals["player_id"], numeric_goals.astype(int)):
            goals_by_player[player_id][gameweek - 1] = int(count)
    return goals_by_player, latest_gameweek


def _read_ownership(path: Path, player_ids: set[str]) -> dict[str, int]:
    ownership = pd.read_csv(path, dtype={"player_id": "string"})
    required = {"player_id", "num_picks"}
    if not required.issubset(ownership.columns):
        raise ValueError(f"Player picks file must contain columns: {sorted(required)}")
    ownership["player_id"] = ownership["player_id"].str.strip()
    if ownership["player_id"].isna().any() or ownership["player_id"].eq("").any():
        raise ValueError("Player picks file contains an empty player_id.")
    if ownership["player_id"].duplicated().any():
        raise ValueError("Player picks file contains duplicate player_id values.")
    unknown_ids = set(ownership["player_id"]) - player_ids
    if unknown_ids:
        raise ValueError(f"Player picks file contains unknown IDs: {sorted(unknown_ids)}")
    counts = pd.to_numeric(ownership["num_picks"], errors="coerce")
    if counts.isna().any() or (counts < 0).any() or (counts % 1 != 0).any():
        raise ValueError("Player picks file contains an invalid num_picks value.")
    return dict(zip(ownership["player_id"], counts.astype(int)))


def _read_prior_season_goals(path: Path, player_ids: set[str]) -> dict[str, int]:
    prior = pd.read_csv(path, dtype={"season": "string", "player_id": "string"})
    required = {"season", "player_id", "last_season_goals", "normalized_name"}
    if not required.issubset(prior.columns):
        raise ValueError(f"ESPN scorer frame must contain columns: {sorted(required)}")
    if prior.empty or not prior["season"].eq(LAST_SEASON).all():
        raise ValueError(f"ESPN scorer frame must contain prior-season {LAST_SEASON} data.")
    names = prior["normalized_name"].astype("string").str.strip()
    if names.isna().any() or names.eq("").any() or names.duplicated().any():
        raise ValueError("ESPN scorer frame contains empty or duplicate normalized names.")
    counts = pd.to_numeric(prior["last_season_goals"], errors="coerce")
    if counts.isna().any() or (counts <= 0).any() or (counts % 1 != 0).any():
        raise ValueError("ESPN scorer frame contains an invalid goals value.")
    matched = prior.loc[prior["player_id"].notna()].copy()
    matched["player_id"] = matched["player_id"].str.strip()
    if matched["player_id"].eq("").any() or matched["player_id"].duplicated().any():
        raise ValueError("ESPN scorer frame contains empty or duplicate player IDs.")
    unknown_ids = set(matched["player_id"]) - player_ids
    if unknown_ids:
        raise ValueError(f"ESPN scorer frame contains unknown IDs: {sorted(unknown_ids)}")
    return dict(zip(matched["player_id"], counts.loc[matched.index].astype(int)))


def build_player_description_facts(data_dir: Path, source: str) -> pd.DataFrame:
    input_dir = data_dir / "input" / source
    intermediate_dir = data_dir / "intermediate" / source
    output_dir = data_dir / "output" / source
    reference = pd.read_csv(input_dir / "player_reference.csv", dtype={"player_id": "string"})
    required = {"player_id", "full_name", "team"}
    if not required.issubset(reference.columns):
        raise ValueError(f"Player reference must contain columns: {sorted(required)}")
    reference = reference[["player_id", "full_name", "team"]].copy()
    for column in ("player_id", "full_name", "team"):
        reference[column] = reference[column].astype("string").str.strip()
    if reference.empty or reference.isna().any().any() or reference.eq("").any().any():
        raise ValueError("Player reference must contain non-empty IDs, names, and teams.")
    if reference["player_id"].duplicated().any():
        raise ValueError("Player reference contains duplicate player_id values.")
    player_ids = set(reference["player_id"])
    goal_history, latest_gameweek = _read_goal_history(input_dir / "player_goals", player_ids)
    owners = _read_ownership(output_dir / "player_pics.csv", player_ids)
    prior_goals = _read_prior_season_goals(intermediate_dir / "espn_goals.csv", player_ids)

    rows = []
    for player in reference.itertuples(index=False):
        player_id = player.player_id
        weekly = goal_history[player_id]
        total = sum(weekly)
        owner_count = owners.get(player_id, 0)
        if total == 0 and owner_count == 0:
            continue
        scoring_weeks = [week for week, count in enumerate(weekly, 1) if count > 0]
        latest_scoring_week = scoring_weeks[-1] if scoring_weeks else None
        rows.append({
            "data_source": source,
            "player_id": player_id,
            "full_name": player.full_name,
            "team": player.team,
            "current_season_goals": total,
            "completed_gameweeks": latest_gameweek,
            "goals_by_gameweek": json.dumps(weekly),
            "scoring_gameweeks": json.dumps(scoring_weeks),
            "brace_gameweeks": json.dumps([week for week, count in enumerate(weekly, 1) if count == 2]),
            "multi_goal_gameweeks": json.dumps([{"gameweek": week, "goals": count} for week, count in enumerate(weekly, 1) if count > 1]),
            "gameweeks_since_last_goal": latest_gameweek - latest_scoring_week if latest_scoring_week is not None else None,
            "ownership_count": owner_count,
            "last_season": LAST_SEASON,
            "last_season_goals": prior_goals.get(player_id),
        })
    if not rows:
        raise ValueError("No players with recorded goals or picks were found.")
    scored_totals = [row["current_season_goals"] for row in rows if row["current_season_goals"] > 0]
    picked_counts = [row["ownership_count"] for row in rows if row["ownership_count"] > 0]
    for row in rows:
        goals = row["current_season_goals"]
        picks = row["ownership_count"]
        row["current_season_goal_rank"] = 1 + sum(other > goals for other in scored_totals) if goals > 0 else None
        row["current_season_goal_rank_tie_count"] = scored_totals.count(goals) if goals > 0 else None
        row["ownership_rank"] = 1 + sum(other > picks for other in picked_counts) if picks > 0 else None
        row["ownership_rank_tie_count"] = picked_counts.count(picks) if picks > 0 else None
    facts = pd.DataFrame(rows)
    if facts["player_id"].duplicated().any():
        raise ValueError("Prepared player facts contain duplicate player IDs.")
    return facts


def _json_value(value: Any) -> Any:
    if value is None or (isinstance(value, numbers.Real) and math.isnan(value)):
        return None
    if isinstance(value, numbers.Integral):
        return int(value)
    if isinstance(value, numbers.Real):
        return float(value)
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value


def build_description_prompt(player_facts: Mapping[str, Any], validation_feedback: str | None = None) -> str:
    fields = (
        "full_name", "team", "current_season_goals", "current_season_goal_rank",
        "current_season_goal_rank_tie_count", "completed_gameweeks", "goals_by_gameweek",
        "scoring_gameweeks", "brace_gameweeks", "multi_goal_gameweeks",
        "gameweeks_since_last_goal", "ownership_count", "last_season", "last_season_goals",
    )
    facts = {key: _json_value(player_facts.get(key)) for key in fields if key in player_facts}
    current_goals = facts.get("current_season_goals")
    ownership_count = facts.get("ownership_count")
    goal_rank = facts.get("current_season_goal_rank")
    goal_rank_tie_count = facts.get("current_season_goal_rank_tie_count")
    goal_count_instruction = (
        f"- GOAL-TOTAL CHECK: current_season_goals is exactly {current_goals}. This is the total across all completed gameweeks; do not recalculate it from scoring_gameweeks or goals_by_gameweek. Any current-season goal total you state must match it exactly.\n"
        if current_goals is not None
        else "- The current-season goal total is unknown; do not state a current-season total.\n"
    )
    if ownership_count is None:
        ownership_instruction = "- Ownership is unknown; omit ownership claims.\n"
    elif ownership_count == 0:
        ownership_instruction = "- OWNERSHIP CHECK: ownership_count is exactly zero current-season picks. You may say no managers have picked the player this season; do not claim historical ownership.\n"
    else:
        manager_label = "manager" if ownership_count == 1 else "managers"
        ownership_instruction = (
            f"- OWNERSHIP CHECK: ownership_count is exactly {ownership_count} current-season picks. If you mention ownership, state exactly {ownership_count} {manager_label}; never say nobody, no managers, or no one picked/selected/owns the player.\n"
        )
    if current_goals is not None and current_goals <= 1:
        rank_instruction = f"- SCORING-RANK CHECK: with only {current_goals} current-season goal(s), do not mention or imply any goalscoring rank (including highest/lowest or ordinal claims), even if a rank is present in the facts.\n"
    elif goal_rank is not None:
        rank_instruction = (
            f"- SCORING-RANK CHECK: the player's rank among all current-season scorers is {goal_rank}; "
            f"{goal_rank_tie_count} player(s) share this rank. If mentioned, describe only this overall rank, never a team/club rank, "
            "and do not give a different ordinal. A tie count of 1 means the player is not tied; a tie count above 1 means the player shares the stated rank, not the next rank.\n"
        )
    else:
        rank_instruction = "- Do not mention a goalscoring rank unless the prepared overall current-season rank is present; never infer a team/club rank or substitute ownership rank.\n"
    goal_history = facts.get("goals_by_gameweek")
    scoreless_gameweeks = sum(goals == 0 for goals in goal_history) if isinstance(goal_history, list) else None
    scoreless_instruction = (
        f"- There are exactly {scoreless_gameweeks} current-season gameweeks with no recorded goals; do not state a different count.\n"
        if scoreless_gameweeks is not None
        else "- Do not quantify current-season gameweeks with no recorded goals unless the exact count is supplied.\n"
    )
    prompt = (
        "Write one natural, polished football description of this fantasy-game player using only the facts in the JSON below.\n\n"
        "STYLE:\n"
        "- Aim for 15-35 words, and never exceed 40 words.\n"
        "- Write a complete natural sentence; do not use a fixed clause template.\n"
        "- Select only the most useful supported facts and vary the sentence structure.\n"
        "FACT CHECKS (mandatory; facts override style preferences):\n"
        f"{goal_count_instruction}"
        f"{ownership_instruction}"
        f"{rank_instruction}"
        f"{scoreless_instruction}"
        "- Ownership counts are current-season picks only; do not attach them to last season or claim historical manager picks, which are unknown. Do not invent or name individual managers.\n"
        "- FINAL OWNERSHIP AUDIT: if ownership_count is greater than zero, the sentence must not say or imply nobody/no one/no managers picked, selected, owns, or backs the player. If you cannot state the exact pick count confidently, omit ownership entirely.\n"
        "- FINAL GOAL AUDIT: every current-season goal total or equivalent claim (including no goals, one goal, once, a single goal, braces, or a numeric total) must agree with current_season_goals exactly. Do not derive a total from the gameweek list; if uncertain, omit the total.\n"
        "- A blank prior-season total means unknown, not zero. Never invent or infer a missing prior-season total.\n"
        "- The gameweek count is current-season data only. Never state or imply how many gameweeks a prior season lasted.\n"
        "- A brace means exactly two goals in a gameweek; describe larger tallies accurately.\n"
        "- A team value gives club affiliation only. It does not establish position, lineup, bench status, starts, appearances, minutes, availability, effort, or ability. Do not claim any of these.\n"
        "- No fixture, opponent, scoreline, or match result is supplied. Do not mention or infer one.\n"
        "- The facts contain no pronouns; use the player's name or neutral wording.\n"
        "- Return exactly one sentence, with no heading or explanation.\n\n"
        "PLAYER FACTS (the only source of claims; do not use outside knowledge):\n"
        f"{json.dumps(facts, ensure_ascii=False, allow_nan=False)}"
    )
    if validation_feedback:
        prompt += (
            "\n\nREVISION REQUIRED: Your previous description failed validation: "
            f"{validation_feedback}\n"
            "Rewrite the sentence from the PLAYER FACTS, not from memory or the previous response. "
            "For an ownership error, remove the zero-picks claim and use the exact positive ownership_count or omit ownership. "
            "For a goal-total error, remove the incorrect total and use current_season_goals exactly, or omit the goal total. "
            "Do not repeat or paraphrase the rejected claim."
        )
    return prompt


def _number_value(token: str) -> int | None:
    normalized = token.lower().replace("-", " ")
    if normalized in {"no", "zero"}:
        return 0
    if normalized in {"a", "an"}:
        return 1
    if normalized.isdigit():
        return int(normalized)
    if normalized.endswith(("st", "nd", "rd", "th")) and normalized[:-2].isdigit():
        return int(normalized[:-2])
    values = [NUMBER_WORDS.get(part) for part in normalized.split()]
    if any(value is None for value in values):
        return None
    return sum(value for value in values if value is not None)


def _validate_goal_counts(description: str, player_id: str, player_facts: Mapping[str, Any]) -> None:
    current_goals = _json_value(player_facts.get("current_season_goals"))
    last_goals = _json_value(player_facts.get("last_season_goals"))
    boundaries = list(re.finditer(r"[,;]|\b(?:and|but|though|while|after|following|whereas|despite|although|versus)\b|\b(?:in contrast to|compared (?:with|to))\b|\bvs\.?\b", description, re.IGNORECASE))
    for match in GOAL_COUNT_PATTERN.finditer(description):
        if match.group("count").lower() in {"first", "second", "third", "fourth", "fifth", "sixth", "seventh", "eighth", "ninth", "tenth"}:
            continue
        start = max((item.end() for item in boundaries if item.end() <= match.start()), default=0)
        end = min((item.start() for item in boundaries if item.start() >= match.end()), default=len(description))
        context = description[start:end].lower()
        prior = re.search(r"\b(?:last|prior|previous|preceding)\s+(?:\w+\s+)?(?:season|year)\b", context)
        season_label = player_facts.get("last_season")
        if isinstance(season_label, str) and season_label.lower() in context:
            prior = True
        expected = last_goals if prior else current_goals
        actual = _number_value(match.group("count"))
        if expected is None or actual is None or not float(expected).is_integer() or actual != int(expected):
            season = "prior-season" if prior else "current-season"
            raise ValueError(f"Description for {player_id} gives a goal total that does not match the prepared {season} facts.")


def _validate_scoring_ranks(description: str, player_id: str, player_facts: Mapping[str, Any]) -> None:
    team = _json_value(player_facts.get("team"))
    if isinstance(team, str) and team.strip():
        team_name = re.escape(team.strip())
        rank_words = r"(?:top|highest|first|second|third|fourth|fifth|\d+(?:st|nd|rd|th))"
        team_rank_pattern = re.compile(
            r"\b" + team_name + r"(?:['’]s)?\s+" + rank_words + r"\b"
            r"|\b" + rank_words + r"(?:-highest)?\b[^.;,]{0,40}\b(?:for|among|of)\s+(?:the\s+)?" + team_name + r"\b"
            r"|\b(?:for|among)\s+(?:the\s+)?" + team_name + r"\b[^.;,]{0,40}\b" + rank_words + r"(?:-highest)?\b",
            re.IGNORECASE,
        )
        if team_rank_pattern.search(description):
            raise ValueError(f"Description for {player_id} claims an unsupported team-specific scoring rank.")
    claims = list(RANK_CLAIM_PATTERN.finditer(description))
    tie_pattern = re.compile(
        r"\b(?:tie(?:d)?|share|shares|sharing|joint(?:ly)?)\s+(?:for\s+)?(?P<rank>" + NUMBER_TOKEN + r")"
        r"(?:\s+(?:place|position))?\b",
        re.IGNORECASE,
    )
    tie_claims = list(tie_pattern.finditer(description))
    has_tie_claim = bool(re.search(r"\b(?:tie|tied|share|shares|sharing|joint(?:ly)?)\b", description, re.IGNORECASE))
    others_tied_pattern = re.compile(
        r"\b(?P<count>" + NUMBER_TOKEN + r")\s+others?\s+(?:sharing|share)\s+(?:(?:this|the\s+same)\s+)?(?:rank|place)\b",
        re.IGNORECASE,
    )
    others_tied_claims = list(others_tied_pattern.finditer(description))
    if not claims and not has_tie_claim:
        return
    expected = _json_value(player_facts.get("current_season_goal_rank"))
    if expected is None:
        raise ValueError(f"Description for {player_id} claims an unavailable scoring rank.")
    current_goals = _json_value(player_facts.get("current_season_goals"))
    if current_goals is not None and int(current_goals) <= 1:
        raise ValueError(f"Description for {player_id} uses a repetitive scoring rank for a one-goal-or-less total.")
    for claim in claims:
        actual = _number_value(claim.group("rank") or claim.group("highest"))
        if actual != int(expected):
            raise ValueError(f"Description for {player_id} gives a scoring rank that does not match the prepared facts.")
    tie_count = _json_value(player_facts.get("current_season_goal_rank_tie_count"))
    if has_tie_claim and (tie_count is None or int(tie_count) <= 1):
        raise ValueError(f"Description for {player_id} claims a scoring-rank tie that does not match the prepared facts.")
    if tie_count is not None and int(tie_count) > 1 and claims and not has_tie_claim:
        raise ValueError(f"Description for {player_id} does not acknowledge the prepared scoring-rank tie.")
    for claim in tie_claims:
        actual = _number_value(claim.group("rank"))
        if actual != int(expected):
            raise ValueError(f"Description for {player_id} gives a tied scoring rank that does not match the prepared facts.")
    for claim in others_tied_claims:
        actual = _number_value(claim.group("count"))
        if tie_count is None or actual != int(tie_count) - 1:
            raise ValueError(f"Description for {player_id} gives an incorrect number of players sharing the scoring rank.")


def _validate_unsupported_context(description: str, player_id: str, player_facts: Mapping[str, Any]) -> None:
    named_manager = re.compile(
        r"\b(?:managers?|owners?)\b\s*(?::|named)?\s*(?:['\"][^'\"]+['\"]|[A-Z][A-Za-z-]*(?:\s+[A-Z][A-Za-z-]*)*)"
    )
    if named_manager.search(description):
        raise ValueError(f"Description for {player_id} names an unsupported manager or owner.")
    unsupported_match_facts = re.compile(
        r"\b(?:draw(?:n)?|win(?:s|ning)?|won|lose(?:s|ing)?|lost|loss|beat(?:s|ing|en)?|"
        r"defeat(?:s|ed|ing)?|opponents?|fixtures?|matches?)\b|\b(?<!\d)\d{1,2}\s*[-–]\s*\d{1,2}(?!\d)\b",
        re.IGNORECASE,
    )
    if unsupported_match_facts.search(description):
        raise ValueError(f"Description for {player_id} includes an unsupported fixture or match-result claim.")
    gameweek_counts = re.compile(r"\b(?P<count>" + NUMBER_TOKEN + r")\s+(?:completed\s+)?gameweeks?\b", re.IGNORECASE)
    prior_season = re.compile(r"\b(?:last|prior|previous|preceding)\s+(?:\w+\s+)?(?:season|year)\b", re.IGNORECASE)
    season_label = player_facts.get("last_season")
    boundaries = list(re.finditer(r"[,;]|\b(?:but|though|while|after|following|whereas|despite|although|versus)\b", description, re.IGNORECASE))
    for match in gameweek_counts.finditer(description):
        start = max((item.end() for item in boundaries if item.end() <= match.start()), default=0)
        end = min((item.start() for item in boundaries if item.start() >= match.end()), default=len(description))
        context = description[start:end]
        if prior_season.search(context) or (isinstance(season_label, str) and season_label.lower() in context.lower()):
            raise ValueError(f"Description for {player_id} gives an unsupported prior-season gameweek count.")
    scoreless_weeks = re.compile(
        r"\b(?P<count>" + NUMBER_TOKEN + r")\s+(?:of\s+(?:those\s+)?)?(?:completed\s+)?gameweeks?\b"
        r"\s+(?:that\s+)?(?:featur(?:ed|ing)|with|having|contained|had|recorded|without|were)\s+(?:no\s+|zero\s+)?(?:goals?|goalless)\b",
        re.IGNORECASE,
    )
    goal_history = _json_value(player_facts.get("goals_by_gameweek"))
    if isinstance(goal_history, list):
        expected_scoreless = sum(goals == 0 for goals in goal_history)
        for match in scoreless_weeks.finditer(description):
            if _number_value(match.group("count")) != expected_scoreless:
                raise ValueError(f"Description for {player_id} gives an incorrect count of scoreless gameweeks.")
    unsupported_context = re.compile(
        r"\b(?:on the bench|benched|substitute|starter|starting XI|starting lineup|line[- ]?up|"
        r"forward|midfielder|defender|striker|goalkeeper|winger)\b",
        re.IGNORECASE,
    )
    if unsupported_context.search(description):
        raise ValueError(f"Description for {player_id} claims unsupported position or lineup information.")


def _claims_historical_ownership(description: str) -> bool:
    for clause in re.split(r"[,;]|\b(?:but|although|while|though)\b", description, flags=re.IGNORECASE):
        if (
            re.search(r"\bmanagers?\b", clause, re.IGNORECASE)
            and re.search(r"\b(?:picked|selected|chosen|owned|backed)\b", clause, re.IGNORECASE)
            and re.search(r"\b(?:last|previous|prior)\s+season\b", clause, re.IGNORECASE)
        ):
            return True
    return False


def _claims_zero_ownership(description: str) -> bool:
    claim_pattern = re.compile(
        r"\b(?:no managers?|nobody|no one)\s+(?:(?:has|have|did|does|do)\s+)?"
        r"(?:ever\s+)?(?:picked|selected|chosen|own(?:s|ed|ing)?|back(?:s|ed|ing)?|support(?:s|ed|ing)?)\b",
        re.IGNORECASE,
    )
    return bool(claim_pattern.search(description))


def validate_description(description: Any, player_id: str, player_facts: Mapping[str, Any]) -> str:
    if not isinstance(description, str):
        raise ValueError(f"Ollama returned no description string for {player_id}.")
    description = re.sub(r"\s+", " ", description).strip()
    if not description:
        raise ValueError(f"Ollama returned an empty description for {player_id}.")
    if len(re.findall(r"[.!?](?=\s|$)", description)) != 1:
        raise ValueError(f"Description for {player_id} must contain exactly one sentence.")
    if len(description.split()) > MAX_DESCRIPTION_WORDS:
        raise ValueError(f"Description for {player_id} exceeds {MAX_DESCRIPTION_WORDS} words.")
    if re.search(r"\b(?:ownership|owner(?:ship)?)\s+(?:rank|ranking|position)\b", description, re.IGNORECASE):
        raise ValueError(f"Description for {player_id} contains an unsupported ownership rank.")
    if _claims_historical_ownership(description):
        raise ValueError(f"Description for {player_id} makes an unsupported historical ownership claim.")
    _validate_unsupported_context(description, player_id, player_facts)
    _validate_scoring_ranks(description, player_id, player_facts)
    _validate_goal_counts(description, player_id, player_facts)
    for match in re.finditer(r"\b(?P<count>" + NUMBER_TOKEN + r")\s+(?:managers?|owners?)\b", description, re.IGNORECASE):
        actual = _number_value(match.group("count"))
        if actual != int(player_facts["ownership_count"]):
            raise ValueError(f"Description for {player_id} gives an ownership count that does not match the prepared facts.")
    if _claims_zero_ownership(description) and int(player_facts["ownership_count"]) != 0:
        raise ValueError(f"Description for {player_id} says nobody picked the player despite nonzero ownership.")
    return description


def build_fallback_description(player_facts: Mapping[str, Any]) -> str:
    player_name = player_facts.get("full_name")
    current_goals = _json_value(player_facts.get("current_season_goals"))
    completed_gameweeks = _json_value(player_facts.get("completed_gameweeks"))
    ownership_count = _json_value(player_facts.get("ownership_count"))
    if not isinstance(player_name, str) or not player_name.strip():
        raise ValueError("Cannot build a fallback description without a player name.")
    if current_goals is None or completed_gameweeks is None or ownership_count is None:
        raise ValueError(f"Cannot build a fallback description without complete current-season facts for {player_name}.")
    goal_label = "goal" if int(current_goals) == 1 else "goals"
    description = (
        f"{player_name} has scored {int(current_goals)} {goal_label} in "
        f"{int(completed_gameweeks)} gameweeks this season"
    )
    if int(ownership_count) == 0:
        description += "; no managers have picked the player this season"
    else:
        manager_label = "manager" if int(ownership_count) == 1 else "managers"
        description += f", with {int(ownership_count)} {manager_label} backing the player"
    description += "."
    return validate_description(description, str(player_facts.get("player_id", "<unknown>")), player_facts)


def generate_player_description(player_facts: Mapping[str, Any], model: str, ollama_url: str, validation_feedback: str | None = None) -> str:
    player_id = str(player_facts.get("player_id", "<unknown>"))
    request_body = {
        "model": model,
        "prompt": build_description_prompt(player_facts, validation_feedback),
        "stream": False,
        "format": {"type": "object", "properties": {"description": {"type": "string"}}, "required": ["description"]},
        "options": {"temperature": 0.4, "top_p": 0.9, "repeat_penalty": 1.12, "num_predict": 120},
    }
    request = urllib.request.Request(ollama_url, data=json.dumps(request_body).encode("utf-8"), headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(request, timeout=180) as response:
            result = json.loads(response.read().decode("utf-8"))
    except urllib.error.URLError as error:
        raise RuntimeError("Could not reach local Ollama. Start Ollama and confirm the model is installed.") from error
    except (UnicodeDecodeError, json.JSONDecodeError) as error:
        raise ValueError("Ollama returned an invalid response.") from error
    try:
        generated = json.loads(result["response"])
    except (KeyError, TypeError, json.JSONDecodeError) as error:
        raise ValueError(f"Ollama returned invalid JSON for {player_id}.") from error
    if not isinstance(generated, dict):
        raise ValueError(f"Ollama returned an invalid description object for {player_id}.")
    return validate_description(generated.get("description"), player_id, player_facts)


## Prepare player facts

All source-data loading, validation, joins, and calculations happen in this notebook before generation. No separate player-description Python script is required.


In [4]:
facts = build_player_description_facts(DATA_DIR, data_source)
if MAX_PLAYERS_TO_GENERATE is not None:
    if isinstance(MAX_PLAYERS_TO_GENERATE, bool) or not isinstance(MAX_PLAYERS_TO_GENERATE, int) or MAX_PLAYERS_TO_GENERATE < 1:
        raise ValueError("MAX_PLAYERS_TO_GENERATE must be a positive integer or None.")
    facts = facts.head(MAX_PLAYERS_TO_GENERATE).copy()
    print(f"Limited run: generating {len(facts)} descriptions; the output CSV will contain only this subset.")
expected_player_ids = facts["player_id"].tolist()
print(f"Prepared and validated facts for {len(expected_player_ids)} players.")
facts[["player_id", "full_name", "current_season_goals", "ownership_count", "last_season_goals", "current_season_goal_rank"]].head(10)


Prepared and validated facts for 95 players.


,player_id,full_name,current_season_goals,ownership_count,last_season_goals,current_season_goal_rank
0,WSL_0018,Alessia Russo,2,4,13.0,18.0
1,WSL_0016,Chloe Kelly,0,1,5.0,NaN
2,WSL_0344,Georgia Stanway,1,4,NaN,40.0
3,WSL_0012,Kim Little,0,1,NaN,NaN
4,WSL_0353,Lisa Baum,1,1,NaN,40.0
5,WSL_0011,Mariona Caldentey,0,2,4.0,NaN
6,WSL_0020,Michelle Agyemang,0,1,1.0,NaN
7,WSL_0015,Olivia Smith,1,2,5.0,40.0
8,WSL_0348,Selina Cerci,0,1,NaN,NaN
9,WSL_0009,Smilla Holmberg,1,0,2.0,40.0


## Generate descriptions

The model writes each complete sentence from that player's verified facts. Each description is validated before it is accepted. After three rejected model responses, the notebook uses a short fact-only fallback and validates that too. The cell reports per-player progress; no CSV is replaced unless every player has a valid description.


In [ ]:
if globals().get("DESCRIPTION_VALIDATOR_VERSION") != 10:
    raise RuntimeError("The player-description validator is out of date. Restart the notebook kernel and run all cells from the top.")

generated_rows = []
generation_issues = []
total_players = len(facts)
for index, player_facts in enumerate(facts.to_dict(orient="records"), start=1):
    player_name = player_facts["full_name"]
    print(f"[{index}/{total_players}] Generating {player_name}...", flush=True)
    validation_feedback = None
    for attempt in range(1, MAX_VALIDATION_ATTEMPTS + 1):
        try:
            description = generate_player_description(
                player_facts,
                model=MODEL,
                ollama_url=OLLAMA_URL,
                validation_feedback=validation_feedback,
            )
            break
        except ValueError as error:
            validation_feedback = str(error)
            if attempt == MAX_VALIDATION_ATTEMPTS:
                try:
                    description = build_fallback_description(player_facts)
                except ValueError as fallback_error:
                    generation_issues.append({
                        "player_id": player_facts["player_id"],
                        "full_name": player_name,
                        "issue": f"Model validation failed: {validation_feedback}; fallback failed: {fallback_error}",
                        "attempts": attempt,
                    })
                    print(f"[{index}/{total_players}] Validation failed after {attempt} attempts; fallback also failed: {fallback_error}", flush=True)
                    continue
                print(f"[{index}/{total_players}] Model validation failed after {attempt} attempts; using verified fact-only fallback: {description}", flush=True)
                break
    else:
        continue
    generated_rows.append({"player_id": player_facts["player_id"], "description": description})
    print(f"[{index}/{total_players}] Complete: {description}", flush=True)

generated_player_ids = [row["player_id"] for row in generated_rows]
if len(set(generated_player_ids)) != len(generated_player_ids) or any(player_id not in expected_player_ids for player_id in generated_player_ids):
    raise ValueError("Generated descriptions do not match prepared player IDs exactly.")
descriptions = [row["description"] for row in generated_rows]
print(f"Generated and validated {len(descriptions)} of {total_players} descriptions.")
if generation_issues:
    print(f"{len(generation_issues)} description issue(s) will be listed in the final output cell.")


[1/95] Generating Alessia Russo...


[1/95] Complete: Alessia Russo scores 2 goals this season, ranking 18th among all current-season scorers, with 4 managers backing her as one of their picks.
[2/95] Generating Chloe Kelly...
[2/95] Complete: Chloe Kelly is currently owned by 1 manager at Arsenal with no goals scored in any of the 8 completed gameweeks this season.
[3/95] Generating Georgia Stanway...
[3/95] Complete: Georgia Stanway scored 1 goal in the current season for Arsenal.
[4/95] Generating Kim Little...
[4/95] Complete: Kim Little has scored zero goals in the current season across eight gameweeks, with no recorded scores in any of those weeks.
[5/95] Generating Lisa Baum...
[5/95] Complete: Lisa Baum scored once in the current season for Arsenal, but failed to find the back of the net in seven gameweeks.
[6/95] Generating Mariona Caldentey...
[6/95] Complete: Mariona Caldentey is owned by 2 managers: [Manager1] and [Manager2], but has scored no goals this season in all 8 gameweeks.
[7/95] Generating Michelle Ag

## Save the descriptions

After every generated row has passed validation, this cell overwrites `player_descriptions.csv` for the configured data source. If generation or validation fails, this cell does not run and the existing CSV stays unchanged.


In [ ]:
if generation_issues:
    print(f"No CSV was written: {len(generation_issues)} player description(s) failed validation. The existing output file is unchanged.")
    display(pd.DataFrame(generation_issues))
else:
    if len(descriptions) != len(facts) or len(set(expected_player_ids)) != len(facts):
        raise ValueError("Description output failed player count or ID validation.")

    output = facts.rename(columns={"current_season_goals": "goals", "completed_gameweeks": "through_gameweek", "ownership_count": "num_picks"}).copy()
    output["description"] = descriptions
    output["generated_at"] = datetime.now().astimezone().isoformat(timespec="seconds")
    output["model"] = MODEL
    if output["player_id"].tolist() != expected_player_ids or output["player_id"].duplicated().any() or output["description"].isna().any() or output["description"].str.strip().eq("").any():
        raise ValueError("Description CSV failed completeness or uniqueness validation.")
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    output.to_csv(OUTPUT_PATH, index=False)
    print(f"Overwrote {len(output)} player descriptions in {OUTPUT_PATH}")
    display(output[["player_id", "full_name", "num_picks", "goals", "last_season_goals", "description"]])


No CSV was written: 7 player description(s) failed validation. The existing output file is unchanged.


,player_id,full_name,issue,attempts
0,WSL_0011,Mariona Caldentey,Description for WSL_0011 says nobody picked th...,3
1,WSL_0020,Michelle Agyemang,Description for WSL_0020 says nobody picked th...,3
2,WSL_0348,Selina Cerci,Description for WSL_0348 says nobody picked th...,3
3,WSL_0043,Chasity Grant,Description for WSL_0043 gives a goal total th...,3
4,WSL_0038,Rachel Daly,Description for WSL_0038 says nobody picked th...,3
5,WSL_0181,Mia Enderby,Description for WSL_0181 gives a goal total th...,3
6,WSL_0339,Andrea Medina,Description for WSL_0339 gives a goal total th...,3


In [ ]:
list(pd.DataFrame(generation_issues)["issue"])

['Description for WSL_0011 says nobody picked the player despite nonzero ownership.',
 'Description for WSL_0020 says nobody picked the player despite nonzero ownership.',
 'Description for WSL_0348 says nobody picked the player despite nonzero ownership.',
 'Description for WSL_0043 gives a goal total that does not match the prepared current-season facts.',
 'Description for WSL_0038 says nobody picked the player despite nonzero ownership.',
 'Description for WSL_0181 gives a goal total that does not match the prepared current-season facts.',
 'Description for WSL_0339 gives a goal total that does not match the prepared current-season facts.']